# పాఠం 13 - Cognee జ్ఞాన గ్రాఫ్‌లతో ఏజెంట్ మెమరీ


## సెటప్

ఈ నోట్బుక్ [**Cognee**](https://www.cognee.ai/) నోళెడ్జ్ గ్రాఫ్‌లు మరియు **Microsoft Agent Framework** (MAF) ఉపయోగించి పర్సిస్టెంట్ మెమరీతో ఒక తెలివైన **కోడింగ్ అసిస్టెంట్** ను ఎలా నిర్మించాలో చూపిస్తుంది.

Cognee అసంఘటిత టెక్ట్స్ ను వెక్టర్ ఎంబెడ్డింగ్స్ ఆధారిత నిర్మిత, ప్రశ్నించదగిన నోళెడ్జ్ గ్రాఫ్ గా మార్చుతుంది — మీ ఏజెంట్ కి సంపన్నమైన, సంబంధం గుర్తించగల దీర్ఘకాలిక జ్ఞాపకాన్ని ఇస్తుంది.

### మీరు నేర్చుకునేది ఏమిటి
1. **నోళెడ్జ్ గ్రాఫ్‌లు తయారు చేయడం**: డెవలపర్ ప్రొఫైల్‌లు మరియు ఉత్తమ ఆచారం ను నిర్మిత, ప్రశ్నించదగిన జ్ఞానం గా మార్చడం.
2. **Cognee ని MAF తో మేళవించడం**: MAF ఏజెంట్ Cognee యొక్క నోళెడ్జ్ గ్రాఫ్ ను ప్రశ్నించడానికి `@tool` ఫంక్షన్లను ఉపయోగించడం.
3. **సెషన్-అ웨어 సంభాషణలు**: ఒకే సెషన్ లో బహుళ ప్రశ్నల సమయంలో సందర్భం నిలుపుకోవడం.
4. **దీర్ఘకాలిక జ్ఞాపకం**: సెషన్ల మధ్య ముఖ్యమైన జ్ఞానాన్ని నిలుపుకోవడం మరియు కొత్త సంభాషణల్లో దాన్ని తిరిగి పొందడం.

### అవసరాలు
- Python 3.9+
- సెషన్ నిర్వహణ కోసం స్థానికంగా Redis నడుస్తోంది (`docker run -d -p 6379:6379 redis`)
- ఒక LLM API కీ (ఉదా. OpenAI) — `.env` లో `LLM_API_KEY` సెట్ చేయాలి
- `.env` లో `CACHING=true` (Cognee సెషన్‌ల కోసం అవసరమే)
- అభివృద్ధి చేసిన చాట్ మోడల్ తో మైక్రోసాఫ్ట్ ఫౌండ్రీ ప్రాజెక్టు
- `.env` లో `AZURE_AI_PROJECT_ENDPOINT` మరియు `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI లో లాగిన్ అయి ఉండాలి (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ఏజెంట్ நினைவుల రకాలు

ఈ నోట్‌బుక్ ప్రధాన పాఠం 13 నోట్‌బుక్ నుండి అదే మూడు మెమరీ రకాలను పరిశీలిస్తుంది, కానీ దీర్ఘకాలిక మెమరీ బ్యాక్‌ఎండ్‌గా Cogneeని ఉపయోగిస్తుంది:

| మెమరీ రకం | యంత్రాంగం | జీవితం |
|---|---|---|
| **వర్కింగ్** | `agent.create_session()` (MAF) | ఒక సంభాషణ |
| **షార్ట్-టర్మ్** | Cognee సెషన్ క్యాష్ (Redis) | ఒక సెషన్ |
| **లాంగ్-టర్మ్** | Cognee జ్ఞాన గ్రాఫ్ + వెక్టర్లు | శాశ్వతం |

### Cognee యొక్క మెమరీ ఆర్కిటెక్చర్
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## కోగ్ని నిల్వ కోసం సిద్ధం చేయండి


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## భాగం 1 — జ్ఞాన మరియు సమాచార కేంద్రాన్ని నిర్మించడం

మన కోడింగ్ అసిస్టెంట్ కోసం సమగ్ర జ్ఞాన కేంద్రాన్ని సృష్టించడానికి మేము మూడు రకాల డేటాను గ్రహిస్తాము:

1. **డెవలపర్ ప్రొఫైల్** — వ్యక్తిగత నైపుణ్యం మరియు సాంకేతిక నేపథ్యం
2. **Python ఉత్తమ అభ్యాసాలు** — సాధ్విభావాలతో కూడిన Python నియమాలు
3. **చరిత్రాత్మక సంభాషణలు** — డెవలపర్స్ మరియు AI అసిస్టెంట్ల మధ్య గత Q&A సెషన్లు


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## పరిజ్ఞాన గ్రాఫ్‌ను ప్రదర్శించండి

Cognee ఇది తీసుకున్న విభాగాలు మరియు సంబంధాల ఇంటరాక్టివ్ HTML విజువలైజేషన్‌ను రేంటర్ చేయగలదు.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## మెమొరీని మెమిఫైతో సమృద్ధిగా చేయాలి

`memify()` జ్ఞాన గ్రాఫ్‌ను విశ్లేషించి, బుద్ధి పూర్వక నిబంధనలను తయారు చేస్తుంది — నమూనాలు, ఉత్తమ పద్ధతులు, మరియు కాన్సెప్ట్‌ల మధ్య సంబంధాలను గుర్తిస్తాయి.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## భాగం 2 — Cognee టూల్స్‌తో MAF ఏజెంట్

ఇప్పుడు మేము `@tool` ఫంక్షన్లు ద్వారా Cognee యొక్క జ్ఞాన గ్రాఫ్‌ను ప్రశ్నించగల MAF ఏజెంట్‌ను సృష్టిస్తాము. ఇది ఏజెంట్‌కు సెషన్ల ద్వారా సంభాషణీయ సందర్భాన్ని నిలుపుకుంటూ గ్రాఫ్-అవగాహన సేమాంటిక్ సెర్చ్ యొక్క పూర్తి శక్తిని ఉపయోగించడానికి అనుమతిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## సెషన్లతో పని చేస్తున్న మెమరీ

`AgentSession` (`agent.create_session()` ద్వారా సృష్టించబడింది) ఒక సెషన్ లో పని చేసే మెమరీని అందిస్తుంది. ఏజెంట్ ముందువైపు సందేశాలను సూచించగలదు అలాగే Cognee యొక్క దీర్ఘకాలిక జ్ఞాన గ్రాఫ్ ను కూడా ప్రశ్నించగలదు.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## కొత్త సెషన్ — దీర్ఘకాలిక స్మృతి కొనసాగుతుంది

కొత్త సెషన్ ప్రారంభించడం వర్కింగ్ మెమరిని క్లియర్ చేస్తుంది, కానీ Cognee నాలెడ్జ్ గ్రాఫ్ ఇంకా అందుబాటులో ఉంటుంది. ఏజెంట్ పూర్తిగా కొత్త సంభాషణలో అదే దీర్ఘకాలిక జ్ఞానాన్ని పొందగలడు.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

In this notebook you built a coding assistant that combines **MAF's working memory** (`agent.create_session()`) with **Cognee's long-term knowledge graph**.

### What You've Learned
1. **Knowledge graph construction**: Cognee ingests unstructured text and builds a graph + vector memory.
2. **Graph enrichment with memify**: Derived facts and richer relationships on top of your existing graph.
3. **MAF + Cognee integration**: `@tool` functions let an MAF agent query Cognee's graph naturally.
4. **Working memory + long-term memory**: `AgentSession` (via `agent.create_session()`) provides session context while Cognee provides persistent knowledge.
5. **Filtered search with NodeSets**: Target specific subsets of the knowledge graph (e.g. only principles).

### Key Takeaways
- **Cognee** turns raw text into structured, relationship-aware memory — more powerful than a flat vector store.
- **`@tool` functions** bridge MAF agents and external knowledge systems cleanly.
- **`AgentSession`** (via `agent.create_session()`) keeps per-conversation context separate from long-lived knowledge.
- The same knowledge graph serves multiple sessions and agents.

### Real-World Applications
- **Developer copilots**: Code review, incident analysis, architecture assistants
- **Customer-facing copilots**: Support agents over product docs, FAQs, and CRM notes
- **Internal expert copilots**: Policy, legal, or security assistants reasoning over guidelines
- **Unified data layers**: Combine structured and unstructured data into one queryable graph

### Next Steps
- Experiment with temporal awareness in Cognee
- Define an OWL ontology for domain-specific graph quality
- Add user feedback loops to improve retrieval over time
- Scale to multi-agent systems sharing the same Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
